In [1]:
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torch import tensor
from torch.nn.functional import pad

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
spacy_en = spacy.load("en_core_web_sm")
spacy_de = spacy.load("de_core_news_sm")

In [4]:
dataset = load_dataset("bentrevett/multi30k")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [6]:
train_dataset = dataset["train"]
val_dataset = dataset["validation"]

In [7]:
en_list = list(train_dataset["en"]) + list(val_dataset["en"])
en_vocab = sorted(list(set(token.text for text in en_list for token in spacy_en.tokenizer(text))))

In [8]:
en_maximum_length = max(len(spacy_en.tokenizer(text)) for text in en_list)
en_maximum_length

41

In [9]:
de_list = list(train_dataset["de"]) + list(val_dataset["de"])
de_vocab = sorted(list(set(token.text for text in de_list for token in spacy_de.tokenizer(text))))

In [10]:
de_maximum_length = max(len(spacy_de.tokenizer(text)) for text in de_list)
de_maximum_length

44

In [11]:
specials = [
    "<s>",
    "</s>",
    "<blank>",
    "<unk>",
]

In [12]:
def index_to_token(index, lang):
    vocab = en_vocab if lang == "en" else de_vocab
    all_vocab = specials + vocab
    return all_vocab[index]
    
def token_to_index(token, lang):
    vocab = en_vocab if lang == "en" else de_vocab
    all_vocab = specials + vocab

    if token not in all_vocab:
        token = "<unk>"

    return all_vocab.index(token)

In [13]:
def sentence_to_tokens(sentence, lang):
    tokenizer = spacy_en.tokenizer if lang == "en" else spacy_de.tokenizer
    return [
        token_to_index(token.text, lang)
        for token in tokenizer(sentence)
    ]
    
def tokens_to_sentence(tokens, lang):
    return " ".join(
        index_to_token(index, lang)
        for index in tokens
    )    

In [14]:
train_dataset[0]["en"]

'Two young, White males are outside near many bushes.'

In [15]:
sentence_to_tokens(train_dataset[0]["en"], "en")

[1765, 10989, 18, 1872, 6605, 2202, 7279, 7039, 6636, 3010, 20]

In [16]:
tokens_to_sentence([1765, 10989, 18, 1872, 6605, 2202, 7279, 7039, 6636, 3010, 20], "en")

'Two young , White males are outside near many bushes .'

In [17]:
max_padding = 50

In [18]:
def collate_fn(batch):
    bs_id = tensor([0])
    eos_id = tensor([1])
    pad_id = 2
    
    src_list, tgt_list = [], []
    for item in batch:
        _src, _tgt = item["en"], item["de"]
        processed_src = torch.cat([
            bs_id,
            torch.tensor(sentence_to_tokens(_src, "en")),
            eos_id,
        ], dim=0)
        
        processed_tgt = torch.cat([
            bs_id,
            torch.tensor(sentence_to_tokens(_tgt, "de")),
            eos_id,
        ], dim=0)
        
        src_padding = max_padding - len(processed_src)    
        if src_padding < 0:
            raise ValueError(f"Source sequence length {len(processed_src)} exceeds max_padding={max_padding}") 
        src_list.append(pad(processed_src, (0, src_padding), value=pad_id))

        tgt_padding = max_padding - len(processed_tgt)
        if tgt_padding < 0:
            raise ValueError(f"Target sequence length {len(processed_tgt)} exceedsmax_padding={max_padding}") 
        tgt_list.append(pad(processed_tgt, (0, tgt_padding), value=pad_id))
        
    src = torch.stack(src_list)
    tgt = torch.stack(tgt_list)
    
    src.to(device)
    tgt.to(device)
    
    return src, tgt

In [19]:
batch_size = 10

In [20]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

In [21]:
item = next(iter(train_dataloader))

In [22]:
item

(tensor([[    0,  1765, 10989,    18,  1872,  6605,  2202,  7279,  7039,  6636,
           3010,    20,     1,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2],
         [    0,  1516,  6750,  5914,  5585,  5601,  2202,  7212,  1913,  5276,
           8029,  9861,    20,     1,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2],
         [    0,   102,  6464,  5289,  3449,  6027,  1913, 10901,  7731,    20,
              1,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2, 

In [23]:
item[0].shape

torch.Size([10, 50])

In [24]:
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

In [25]:
next(iter(val_dataloader))

(tensor([[    0,   102,  5456,  7165,  6750,  2202,  6477,  3780,  7201,  1913,
          10315,     1,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2],
         [    0,   102,  6609,  9118,  5914,  1913,  5415,  8480,  7188,  1913,
           3781,    20,     1,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2,     2,     2,     2,     2,     2,     2],
         [    0,   102,  2831, 10742,  5630,  9040,  7188,  1913, 10890,    15,
           8940,    20,     1,     2,     2,     2,     2,     2,     2,     2,
              2,     2,     2,     2, 

In [26]:
tokens_to_sentence([ 102,  5456,  7165,  6750,  2202,  6477,  3780,  7201,  1913, 10315], "en")

'A group of men are loading cotton onto a truck'

In [27]:
src_vocab = len(en_vocab) + len(specials)
tgt_vocab = len(de_vocab) + len(specials)
src_vocab, tgt_vocab

(11010, 19621)